# Wybór modelu

In [1]:
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import ComplementNB, MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
import joblib

## Wczytywanie przygotowanych danych

In [2]:
base_path = Path("..") / "data" 
vectorized_dir = base_path / "vectorized"
models_dir = base_path / "models"

vectorizer: TfidfVectorizer = joblib.load(
        vectorized_dir / "tfidf_vectorizer.pkl")

X_train_tfidf, X_test_tfidf, y_train, y_test = joblib.load(
    vectorized_dir / "tfidf_splits.pkl")

In [3]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

print(y_train.shape)
print(y_test.shape)

(4135, 3000)
(1034, 3000)
(4135,)
(1034,)


## Wybór metryk

W celu porównania modeli Naive Bayes wykorzystamy metryki accuracy, precision, recall oraz F1-score.

W przypadku detektora spamu szczególnie istotne jest prawidłowe rozpoznawanie wiadomości należących do klasy `spam`. Dlatego jako główne kryterium porównania przyjmujemy **F1-score dla klasy `spam`**.


## Model odniesienia (DummyClassifier)

> Zanim przejdziemy do zaawansowanych algorytmów, musimy ustalić punkt odniesienia (`baseline`). Użyjemy do tego `DummyClassifier` ze strategią `most_frequent`.

In [4]:
def evaluate_model(model, X_test = X_test_tfidf, y_test = y_test, model_name="-"):
    """
    Function to evaluate a model on the test set and print the results
    """
    print(f"Model name: {model_name}")
    print("-" * 60)
    
    y_pred = model.predict(X_test)

    print("Classification report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("=" * 60 + "\n")

In [5]:
dummy_clf = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_clf.fit(X_train_tfidf, y_train)

evaluate_model(dummy_clf, model_name="DummyClassifier")

Model name: DummyClassifier
------------------------------------------------------------
Classification report:
              precision    recall  f1-score   support

           0       0.87      1.00      0.93       903
           1       0.00      0.00      0.00       131

    accuracy                           0.87      1034
   macro avg       0.44      0.50      0.47      1034
weighted avg       0.76      0.87      0.81      1034

Confusion matrix:
[[903   0]
 [131   0]]



> Powyżej możemy zobaczyć, dlaczego nie warto wybierać `accuracy` jako głównej metryki. Chociaż jest ona dość wysoka, model pominął **100%** wiadomości ze spamem, co oznacza, że zupełnie nie wykonuje swojego zadania.

## Porównanie modeli z domyślnymi parametrami

In [6]:
def compare_models(
        models,
        X_train_tfidf=X_train_tfidf, 
        X_test_tfidf=X_test_tfidf, 
        y_train=y_train, 
        y_test=y_test):

    best_model = None
    best_score = 0.0
    best_model_name = ""

    print("--- COMPARISON OF MODELS ---")

    for name, model in models.items():
        model.fit(X_train_tfidf, y_train)
        y_pred = model.predict(X_test_tfidf)

        print("Confusion matrix:")
        print(confusion_matrix(y_test, y_pred))

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"Model: {name:<25} | Accuracy: {acc:.4f} | F1-score: {f1:.4f}")

        if f1 > best_score:
            best_model = model
            best_score = f1
            best_model_name = name

        print("-" * 40)

    print(f"\nBest model: {best_model_name} (F1 score: {best_score:.4f})")

    return best_model, best_score, best_model_name

default_models = {
    "Complement Naive Bayes": ComplementNB(),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Bernoulli Naive Bayes": BernoulliNB(),
    "Logistic Regression": LogisticRegression(random_state=42),
    "Linear SVM (LinearSVC)": LinearSVC(random_state=42),
    "SGD Classifier": SGDClassifier(random_state=42)
}

default_best_model, default_best_score, default_best_model_name = compare_models(default_models)

--- COMPARISON OF MODELS ---
Confusion matrix:
[[855  48]
 [  5 126]]
Model: Complement Naive Bayes    | Accuracy: 0.9487 | F1-score: 0.8262
----------------------------------------
Confusion matrix:
[[903   0]
 [ 27 104]]
Model: Multinomial Naive Bayes   | Accuracy: 0.9739 | F1-score: 0.8851
----------------------------------------
Confusion matrix:
[[903   0]
 [ 13 118]]
Model: Bernoulli Naive Bayes     | Accuracy: 0.9874 | F1-score: 0.9478
----------------------------------------
Confusion matrix:
[[901   2]
 [ 40  91]]
Model: Logistic Regression       | Accuracy: 0.9594 | F1-score: 0.8125
----------------------------------------
Confusion matrix:
[[901   2]
 [ 18 113]]
Model: Linear SVM (LinearSVC)    | Accuracy: 0.9807 | F1-score: 0.9187
----------------------------------------
Confusion matrix:
[[899   4]
 [ 16 115]]
Model: SGD Classifier            | Accuracy: 0.9807 | F1-score: 0.9200
----------------------------------------

Best model: Bernoulli Naive Bayes (F1 score: 0.9478)

> Ze wszystkich przetestowanych modeli z domyślnymi parametrami, najlepiej prezentuje się `Bernoulli NB` z `f1_score=0.9478`

> W następnym kroku wybierzmy modele z najlepszymi wynikami i przetestujmy, czy dopasowanie parametrów pomoże otrzymać lepsze wyniki. Wybierzemy `Bernoulli Naive Bayes`, `SGD Classifier` i `Linear SVM (LinearSVC)`.

## Dopasowanie parametrów do modeli

In [7]:
tuned_models = {
    # Bernoulli Naive Bayes
    "Bernoulli NB (alpha=0.01)": BernoulliNB(alpha=0.01),
    "Bernoulli NB (alpha=0.1)": BernoulliNB(alpha=0.1),
    "Bernoulli NB (alpha=0.5)": BernoulliNB(alpha=0.5),
    "Bernoulli NB (alpha=1.0)": BernoulliNB(alpha=1.0),
    "Bernoulli NB (alpha=2.0)": BernoulliNB(alpha=2.0),
    "Bernoulli NB (alpha=1.0, binarize=0.2)": BernoulliNB(alpha=1.0, binarize=0.2),
    
    # SGD Classifier
    "SGD (alpha=1e-5, loss='log_loss')": SGDClassifier(alpha=1e-5, loss='log_loss', random_state=42),
    "SGD (alpha=1e-4, loss='hinge')": SGDClassifier(alpha=1e-4, loss='hinge', random_state=42),
    "SGD (alpha=1e-3, loss='modified_huber')": SGDClassifier(alpha=1e-3, loss='modified_huber', random_state=42),
    "SGD (alpha=0.01)": SGDClassifier(alpha=0.01, random_state=42),

    # Linear SVM (LinearSVC)
    "Linear SVM (C=0.01, max_iter=2000)": LinearSVC(C=0.01, random_state=42, max_iter=2000),
    "Linear SVM (C=0.1, max_iter=2000)": LinearSVC(C=0.1, random_state=42, max_iter=2000),
    "Linear SVM (C=1.0, max_iter=2000)": LinearSVC(C=1.0, random_state=42, max_iter=2000),
    "Linear SVM (C=10.0, max_iter=2000)": LinearSVC(C=10.0, random_state=42, max_iter=2000),
    "Linear SVM (C=1.0, loss='hinge', max_iter=2000)": LinearSVC(C=1.0, loss='hinge', random_state=42, max_iter=2000),
}

tuned_best_model, tuned_best_score, tuned_best_name = compare_models(tuned_models)

--- COMPARISON OF MODELS ---
Confusion matrix:
[[903   0]
 [ 12 119]]
Model: Bernoulli NB (alpha=0.01) | Accuracy: 0.9884 | F1-score: 0.9520
----------------------------------------
Confusion matrix:
[[903   0]
 [  9 122]]
Model: Bernoulli NB (alpha=0.1)  | Accuracy: 0.9913 | F1-score: 0.9644
----------------------------------------
Confusion matrix:
[[903   0]
 [ 12 119]]
Model: Bernoulli NB (alpha=0.5)  | Accuracy: 0.9884 | F1-score: 0.9520
----------------------------------------
Confusion matrix:
[[903   0]
 [ 13 118]]
Model: Bernoulli NB (alpha=1.0)  | Accuracy: 0.9874 | F1-score: 0.9478
----------------------------------------
Confusion matrix:
[[902   1]
 [ 28 103]]
Model: Bernoulli NB (alpha=2.0)  | Accuracy: 0.9720 | F1-score: 0.8766
----------------------------------------
Confusion matrix:
[[902   1]
 [ 17 114]]
Model: Bernoulli NB (alpha=1.0, binarize=0.2) | Accuracy: 0.9826 | F1-score: 0.9268
----------------------------------------
Confusion matrix:
[[900   3]
 [ 17 114]]

## Zapisywanie najlepszego modelu

In [8]:
best_model = default_best_model if default_best_score > tuned_best_score else tuned_best_model
best_score = default_best_score if default_best_score > tuned_best_score else tuned_best_score
best_model_name = default_best_model_name if default_best_score > tuned_best_score else tuned_best_name

print(f"\nBest model: {best_model_name} (F1 score: {best_score:.4f})")


Best model: Bernoulli NB (alpha=0.1) (F1 score: 0.9644)


In [9]:
evaluate_model(best_model, model_name=best_model_name)

Model name: Bernoulli NB (alpha=0.1)
------------------------------------------------------------
Classification report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       903
           1       1.00      0.93      0.96       131

    accuracy                           0.99      1034
   macro avg       1.00      0.97      0.98      1034
weighted avg       0.99      0.99      0.99      1034

Confusion matrix:
[[903   0]
 [  9 122]]



In [10]:
joblib.dump(
    best_model,
    models_dir / "spam_classifier_model.pkl"
)

['..\\data\\models\\spam_classifier_model.pkl']

## Podsumowanie

* **Najlepszy model:** `Bernoulli NB (alpha=0.1)` okazał się najlepszym (**Accuracy: 0.9913**, **F1-score: 0.9644**).
* **Macierz pomyłek:** `[[903, 0], [9, 122]]`
* **Zero fałszywych alarmów (FP = 0):** Żadna legalna wiadomość nie została błędnie oznaczona jako spam, co chroni ważne wiadomości użytkownika.
* **Wysoka skuteczność:** Model poprawnie wychwycił 122 wiadomości spamowe, przepuszczając zaledwie 9.

Zastosowanie odpowiedniego algorytmu oraz zapisanie wektoryzatora i modelu kończy proces trenowania, umożliwiając łatwą integrację z docelową aplikacją.